In [2]:
!pip install transformers datasets evaluate seqeval torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00


In [3]:
import xml.etree.ElementTree as ET
import json
import os
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict, ClassLabel, Sequence, Features, Value
import pandas as pd

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

torch: 2.10.0+cu128
CUDA available: True


In [4]:
from pathlib import Path

DATA_DIR    = Path("./datasets")
MAMS_DIR    = DATA_DIR / "mams"
SEMEVAL_DIR = DATA_DIR / "semeval2014"
OUTPUT_DIR  = Path("./prepared_data")
OUTPUT_DIR.mkdir(exist_ok=True)

ASE_DIR  = OUTPUT_DIR / "ase_dataset"
ABSA_DIR = OUTPUT_DIR / "absa_dataset"

MODEL_NAME  = "bert-base-uncased"
MAX_SEQ_LEN = 128

ASE_LABEL_LIST = ["O", "B-ASP", "I-ASP"]
ASE_LABEL2ID   = {l: i for i, l in enumerate(ASE_LABEL_LIST)}
ASE_ID2LABEL   = {i: l for i, l in enumerate(ASE_LABEL_LIST)}

ABSA_LABEL_LIST = ["positive", "negative", "neutral"]
ABSA_LABEL2ID   = {l: i for i, l in enumerate(ABSA_LABEL_LIST)}
SKIP_POLARITY   = {"conflict"}

In [5]:
import xml.etree.ElementTree as ET

def parse_xml(filepath):
    tree = ET.parse(filepath)
    records = []
    for sentence in tree.getroot().findall("sentence"):
        text_el = sentence.find("text")
        if text_el is None or text_el.text is None:
            continue
        text    = text_el.text
        aspects = []
        asp_terms_el = sentence.find("aspectTerms")
        if asp_terms_el is not None:
            for asp in asp_terms_el.findall("aspectTerm"):
                aspects.append({
                    "term"      : asp.get("term", "").strip(),
                    "char_start": int(asp.get("from")),
                    "char_end"  : int(asp.get("to")),
                    "polarity"  : asp.get("polarity", "").strip().lower(),
                })
        records.append({"text": text, "aspects": aspects})
    return records

semeval_extra = []
for fname in ("laptop.xml", "restaurants.xml"):
    semeval_extra.extend(parse_xml(SEMEVAL_DIR / fname))

raw = {
    "train": parse_xml(MAMS_DIR / "train.xml") + semeval_extra,
    "val"  : parse_xml(MAMS_DIR / "val.xml"),
    "test" : parse_xml(MAMS_DIR / "test.xml"),
}

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
def assign_bio_labels(text, aspects, tokenizer, max_length=128):
    encoding = tokenizer(
        text,
        max_length             = max_length,
        truncation             = True,
        padding                = "max_length",
        return_offsets_mapping = True,
    )
    word_ids       = encoding.word_ids()
    offset_mapping = encoding["offset_mapping"]
    labels         = []
    prev_word_id   = None

    for idx, word_id in enumerate(word_ids):
        if word_id is None:
            labels.append(-100)
            prev_word_id = word_id
            continue
        if word_id == prev_word_id:
            labels.append(-100)
            prev_word_id = word_id
            continue

        token_start, token_end = offset_mapping[idx]
        label = ASE_LABEL2ID["O"]

        for asp in aspects:
            if token_start >= asp["char_start"] and token_end <= asp["char_end"]:
                label = ASE_LABEL2ID["B-ASP"] if token_start == asp["char_start"] else ASE_LABEL2ID["I-ASP"]
                break

        labels.append(label)
        prev_word_id = word_id

    result = {
        "input_ids"     : encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels"        : labels,
    }
    if "token_type_ids" in encoding:
        result["token_type_ids"] = encoding["token_type_ids"]
    return result

In [8]:
from datasets import Dataset, DatasetDict

def build_ase_split(records):
    rows = []
    for r in records:
        enc = assign_bio_labels(r["text"], r["aspects"], tokenizer, MAX_SEQ_LEN)
        enc["text"] = r["text"]
        rows.append(enc)
    return Dataset.from_list(rows)

ase_dataset = DatasetDict({split: build_ase_split(records) for split, records in raw.items()})
ase_dataset.save_to_disk(str(ASE_DIR))

Saving the dataset (0/1 shards):   0%|          | 0/10383 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
def build_absa_split(records):
    rows = []
    for r in records:
        for asp in r["aspects"]:
            if asp["polarity"] in SKIP_POLARITY or asp["polarity"] not in ABSA_LABEL2ID:
                continue
            rows.append({
                "text"       : r["text"],
                "aspect_term": asp["term"],
                "char_start" : asp["char_start"],
                "char_end"   : asp["char_end"],
                "label"      : ABSA_LABEL2ID[asp["polarity"]],
                "label_str"  : asp["polarity"],
            })
    return Dataset.from_list(rows)

absa_dataset = DatasetDict({split: build_absa_split(records) for split, records in raw.items()})
absa_dataset.save_to_disk(str(ABSA_DIR))

Saving the dataset (0/1 shards):   0%|          | 0/17101 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1332 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1336 [00:00<?, ? examples/s]

In [10]:
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification
)
from datasets import load_from_disk
import evaluate
import numpy as np

MODEL_NAME = "bert-base-uncased"
ASE_DIR    = "./prepared_data/ase_dataset"

LABEL_LIST = ["O", "B-ASP", "I-ASP"]
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}

dataset   = load_from_disk(ASE_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels = len(LABEL_LIST),
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [11]:
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    logits, labels = p
    preds = np.argmax(logits, axis=-1)

    true_labels = [[LABEL_LIST[l] for l in row if l != -100] for row in labels]
    true_preds  = [
        [LABEL_LIST[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]
    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": result["overall_precision"],
        "recall":    result["overall_recall"],
        "f1":        result["overall_f1"],
    }

In [12]:
args = TrainingArguments(
    output_dir          = "./ase_model",
    num_train_epochs    = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 3e-5,
    warmup_steps        = 100,
    weight_decay        = 0.01,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "f1",
    logging_steps       = 50,
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["val"],
    processing_class = tokenizer,
    data_collator    = DataCollatorForTokenClassification(tokenizer),
    compute_metrics  = compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.111125,0.133450,0.736439,0.795045,0.764621
2,0.074885,0.136387,0.735333,0.828078,0.778955
3,0.057021,0.153217,0.737158,0.829580,0.780643
4,0.045773,0.179396,0.755386,0.816066,0.784554
5,0.023807,0.199014,0.756320,0.808559,0.781567


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=3245, training_loss=0.07296503504913283, metrics={'train_runtime': 1243.9647, 'train_samples_per_second': 41.733, 'train_steps_per_second': 2.609, 'total_flos': 3391335942071040.0, 'train_loss': 0.07296503504913283, 'epoch': 5.0})

In [13]:
results = trainer.evaluate(dataset["test"])
print(f"Test F1: {results['eval_f1']:.4f}")
print(f"Test Precision: {results['eval_precision']:.4f}")
print(f"Test Recall: {results['eval_recall']:.4f}")

trainer.save_model("./ase_model/best")
tokenizer.save_pretrained("./ase_model/best")

Test F1: 0.7641
Test Precision: 0.7358
Test Recall: 0.7948


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./ase_model/best/tokenizer_config.json', './ase_model/best/tokenizer.json')

In [14]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import load_from_disk
import evaluate
import numpy as np

MODEL_NAME = "bert-base-uncased"
ABSA_DIR   = "./prepared_data/absa_dataset"

LABEL_LIST = ["positive", "negative", "neutral"]
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for i, l in enumerate(LABEL_LIST)}

dataset   = load_from_disk(ABSA_DIR)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = len(LABEL_LIST),
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        batch["aspect_term"],
        truncation = True,
        max_length = 128,
    )

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "token_type_ids", "labels"])

Map:   0%|          | 0/17101 [00:00<?, ? examples/s]

Map:   0%|          | 0/1332 [00:00<?, ? examples/s]

Map:   0%|          | 0/1336 [00:00<?, ? examples/s]

In [16]:
metric = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    acc   = metric.compute(predictions=preds, references=p.label_ids)
    # also compute per-class f1
    f1 = evaluate.load("f1")
    f1_result = f1.compute(predictions=preds, references=p.label_ids, average="macro")
    return {"accuracy": acc["accuracy"], "f1": f1_result["f1"]}

args = TrainingArguments(
    output_dir          = "./absa_model",
    num_train_epochs    = 5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 2e-5,
    warmup_steps        = 100,
    weight_decay        = 0.01,
    eval_strategy       = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end  = True,
    metric_for_best_model   = "f1",
    logging_steps       = 50,
)

trainer = Trainer(
    model            = model,
    args             = args,
    train_dataset    = dataset["train"],
    eval_dataset     = dataset["val"],
    processing_class = tokenizer,
    data_collator    = DataCollatorWithPadding(tokenizer),
    compute_metrics  = compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.475259,0.526265,0.800300,0.793473
2,0.387892,0.520016,0.817568,0.811250
3,0.230316,0.665212,0.811562,0.805989
4,0.154746,0.883566,0.805556,0.801262
5,0.076605,0.979610,0.810811,0.806650


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5345, training_loss=0.30824163336972416, metrics={'train_runtime': 1135.7471, 'train_samples_per_second': 75.285, 'train_steps_per_second': 4.706, 'total_flos': 2676361420992546.0, 'train_loss': 0.30824163336972416, 'epoch': 5.0})

In [17]:
results = trainer.evaluate(dataset["test"])
print(f"Test Accuracy : {results['eval_accuracy']:.4f}")
print(f"Test Macro F1 : {results['eval_f1']:.4f}")

trainer.save_model("./absa_model/best")
tokenizer.save_pretrained("./absa_model/best")

Test Accuracy : 0.8211
Test Macro F1 : 0.8153


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./absa_model/best/tokenizer_config.json', './absa_model/best/tokenizer.json')

In [18]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification

ASE_PATH  = "./ase_model/best"
ABSA_PATH = "./absa_model/best"

ase_tokenizer  = AutoTokenizer.from_pretrained(ASE_PATH)
ase_model      = AutoModelForTokenClassification.from_pretrained(ASE_PATH)

absa_tokenizer = AutoTokenizer.from_pretrained(ABSA_PATH)
absa_model     = AutoModelForSequenceClassification.from_pretrained(ABSA_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ase_model.to(device).eval()
absa_model.to(device).eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [19]:
def extract_aspects(text: str) -> list[str]:
    inputs   = ase_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    word_ids = ase_tokenizer(text, truncation=True, max_length=128).word_ids()

    with torch.no_grad():
        logits = ase_model(**inputs).logits

    preds    = torch.argmax(logits, dim=-1)[0].tolist()
    tokens   = ase_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    id2label = ase_model.config.id2label
    aspects  = []
    current  = []
    prev_word_id = None

    def flush():
        if current:
            term = ""
            for t in current:
                if t.startswith("##"):
                    term += t[2:]
                else:
                    term += ("" if term == "" else " ") + t
            aspects.append(term.strip())
            current.clear()

    for token, pred, word_id in zip(tokens, preds, word_ids):
        if word_id is None:
            flush()
            prev_word_id = None
            continue

        if word_id != prev_word_id:
            label = id2label[pred]
            if label == "B-ASP":
                flush()
                current.append(token)
            elif label == "I-ASP" and current:
                current.append(token)
            else:
                flush()

        prev_word_id = word_id

    flush()
    return [a for a in aspects if a]

In [20]:
def get_sentiment(text: str, aspect: str) -> dict:
    inputs = absa_tokenizer(text, aspect, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = absa_model(**inputs).logits
    probs     = torch.softmax(logits, dim=-1)[0].tolist()
    label_idx = int(np.argmax(probs))
    return {
        "label": absa_model.config.id2label[label_idx],
        "score": round(probs[0], 4),  # always positive class probability
    }

In [21]:
def analyze_reviews(reviews: list[str]) -> dict:
    all_reviews    = []
    aspect_summary = {}
    sentiment_scores = []

    for text in reviews:
        aspects = extract_aspects(text)
        aspect_results = []

        for aspect in aspects:
            sentiment = get_sentiment(text, aspect)
            aspect_results.append({"term": aspect, **sentiment})

            if aspect not in aspect_summary:
                aspect_summary[aspect] = {"positive": 0, "negative": 0, "neutral": 0, "scores": []}
            aspect_summary[aspect][sentiment["label"]] += 1
            aspect_summary[aspect]["scores"].append(sentiment["score"])  # already 0→1 positivity

        if aspect_results:
            review_score = round(float(np.mean([a["score"] for a in aspect_results])), 4)
        else:
            review_score = None

        all_reviews.append({"text": text, "aspects": aspect_results, "review_score": review_score})

        if review_score is not None:
            sentiment_scores.append(review_score)

    for asp in aspect_summary:
        scores = aspect_summary[asp].pop("scores")
        aspect_summary[asp]["avg_score"] = round(float(np.mean(scores)), 4)

    overall_score = round(float(np.mean(sentiment_scores)), 4) if sentiment_scores else None
    label_counts  = {"positive": 0, "negative": 0, "neutral": 0}
    for r in all_reviews:
        for a in r["aspects"]:
            label_counts[a["label"]] += 1

    return {
        "overall_score"  : overall_score,
        "overall_label"  : max(label_counts, key=label_counts.get),
        "summary"        : {**label_counts, "total_aspects": sum(label_counts.values())},
        "reviews"        : all_reviews,
        "aspect_summary" : aspect_summary,
    }

In [22]:
import json

reviews1 = [
    "The battery life is amazing but the screen is terrible.",
    "Great camera and fast delivery, packaging was a bit disappointing.",
    "Solid build quality, the price is fair for what you get.",
]

reviews2 = [
    "Amazing sound quality, totally worth it!",
    "Great quality for the price!",
    "Assembly was a nightmare, instructions are terrible.",
    "Battery life is incredible, love these.",
]


output = analyze_reviews(reviews1)
print(json.dumps(output, indent=2))

output = analyze_reviews(reviews2)
print(json.dumps(output, indent=2))

{
  "overall_score": 0.7187,
  "overall_label": "positive",
  "summary": {
    "positive": 5,
    "negative": 2,
    "neutral": 0,
    "total_aspects": 7
  },
  "reviews": [
    {
      "text": "The battery life is amazing but the screen is terrible.",
      "aspects": [
        {
          "term": "battery life",
          "label": "positive",
          "score": 0.9951
        },
        {
          "term": "screen",
          "label": "negative",
          "score": 0.0031
        }
      ],
      "review_score": 0.4991
    },
    {
      "text": "Great camera and fast delivery, packaging was a bit disappointing.",
      "aspects": [
        {
          "term": "camera",
          "label": "positive",
          "score": 0.9952
        },
        {
          "term": "delivery",
          "label": "positive",
          "score": 0.9936
        },
        {
          "term": "packaging",
          "label": "negative",
          "score": 0.0045
        }
      ],
      "review_score": 0.66

In [23]:
from huggingface_hub import login
login()

In [25]:
from transformers import AutoModelForTokenClassification, AutoModelForSequenceClassification, AutoTokenizer

AutoModelForTokenClassification.from_pretrained("./ase_model/best").push_to_hub("UseCondomsKid/ase-model")
AutoTokenizer.from_pretrained("./ase_model/best").push_to_hub("UseCondomsKid/ase-model")

AutoModelForSequenceClassification.from_pretrained("./absa_model/best").push_to_hub("UseCondomsKid/absa-model")
AutoTokenizer.from_pretrained("./absa_model/best").push_to_hub("UseCondomsKid/absa-model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3y2r4wu/model.safetensors:   0%|          | 14.2kB /  436MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0q6cdf7/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/UseCondomsKid/absa-model/commit/38df9a9248a463ca5bd478a0b8bb6632719db0c4', commit_message='Upload tokenizer', commit_description='', oid='38df9a9248a463ca5bd478a0b8bb6632719db0c4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/UseCondomsKid/absa-model', endpoint='https://huggingface.co', repo_type='model', repo_id='UseCondomsKid/absa-model'), pr_revision=None, pr_num=None)